---
title: "Revenue loss attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. We use Shapley values to asses the contribution of each individual product to a global loss of revenue per sale. In order to make computations treatable we rely on Monte Carlo methods. 
format:
  html:
    code-fold: true
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
---

# Initialization

## Imports and settings

In [1]:
import pandas as pd

In [38]:
pd.options.display.float_format = "{:,.2f}".format

## Auxiliary functions

In [45]:
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df["rps_before"] = df["revenue_before"] / df["sales_before"]
    df["rps_after"] = df["revenue_after"] / df["sales_after"]
    df["rps_diff"] = df["rps_after"] - df["rps_before"]
    df["rps_ratio"] = df["rps_after"] / df["rps_before"]
    df["rps_perc_diff"] = 100 * (df["rps_ratio"] - 1)

    return df

def aggregate_data(df: pd.DataFrame) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum()).T
    df_agg["sales_before"] = df_agg["sales_before"].astype(int)
    df_agg["sales_after"] = df_agg["sales_after"].astype(int)

    return df_agg

# Understanding the business problem

## The problem

In [50]:
df_toy = pd.DataFrame(
    columns=["product", "sales_before", "revenue_before", "sales_after", "revenue_after"],
    data=[
        ["A", 200, 10000.00, 25, 1500.00],
        ["B",  40,   800.00, 40,  900.00]
    ]
)

df_agg = aggregate_data(df_toy)

df_toy = add_derived_columns(df_toy)
df_agg = add_derived_columns(df_agg)

print("Toy example:")
display(df_toy)
print("Aggregated data:")
display(df_agg)

Toy example:


,product,sales_before,revenue_before,sales_after,revenue_after,rps_before,rps_after,rps_diff,rps_ratio,rps_perc_diff
0,A,200,"10,000.00",25,"1,500.00",50.00,60.00,10.00,1.20,20.00
1,B,40,800.00,40,900.00,20.00,22.50,2.50,1.12,12.50


Aggregated data:


,product,sales_before,revenue_before,sales_after,revenue_after,rps_before,rps_after,rps_diff,rps_ratio,rps_perc_diff
0,AB,240,"10,800.00",65,"2,400.00",45.00,36.92,-8.08,0.82,-17.95


Even in a simple example with only two products we can see how the phenomenon of Simpson's paradox arises:

- Both products individual RPS goes up: +20% for product A and +12.5% for product B

- However the aggregated RPS goes down: -17.95%

This example actually illustrates a very common business situation:

- Product A runs under some algorithm that optimizes RPS

- The algorithm discards low RPS sales and the final effect is that both revenue and sales decrease, albeit in a way that RPS increases (in other words: the percentual decrease in revenue is less than the percentual decrease in sales)

- When aggregating the data with other products this has a harming effect in the global RPS

In a real case scenario where we have ~4,000 different products instead of just a couple, these interactions become much more complex.

## The solution

In order to navigate the paradox we borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us each product is a player and the common goal is the aggregated RPS. We want to measure how much each individual product contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result. In our example above, the sum of the Shapley value of A and the Shapley value of B must be -17.95.

*Note:* It is not our intend to provide a detailed account on how Shapley values are computed. In the present section we ask the reader to trust us, while in future more technical sections we assume the reader has enough familiarity with the concept.